# run_all_0418_nb12 — adaptive (switching rule), initial=anchor_high

Part of the 0418 follow-up sweep.

* Mode               : **adaptive**
* Seller strategy    : **switching rule; starts at anchor_high**
* Buyer              : random uniform in [cost, s_prev), every round
* `n_rounds`         : 16 (up from 15 in 0416b)
* Product sampling   : `c, v-c, M-v  iid  Uniform(0, 5)` per run
* Switching rule     : `g = (s_prev − m_t)/s_prev`; `g<0.25`→cooperative, `g>0.6`→anchor_high, else keep
* Target runtime     : **~10.4 h** on `deepseek-reasoner`


In [ ]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI
import exp_runner_0418 as R
from run_all_0418 import run_block


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 16
NOTEBOOK_ID = "nb12"
OUTPUT_ROOT = Path("results_0418")
BASE_SEED   = 17000

INITIAL_STRATEGY = 'anchor_high'
ADAPTIVE         = True
N_RUNS           = 130


## Run the block

In [ ]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_block(
    client=CLIENT, model=MODEL,
    initial_strategy=INITIAL_STRATEGY,
    adaptive=ADAPTIVE,
    n_runs=N_RUNS,
    notebook_id=NOTEBOOK_ID,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,
)
print(f"\nTotal runs saved: {manifest['n_runs_done']}/{manifest['n_runs_requested']}")
print(f"Total time: {manifest['total_minutes']:.1f} minutes")


## Inspect the first completed run

In [ ]:
# ── Quick inspection: load and print the first completed run ────────
import json
from pathlib import Path

out_dir = OUTPUT_ROOT / NOTEBOOK_ID
files = sorted(out_dir.glob("run_*.json"))
print(f"{len(files)} run JSONs saved in {out_dir}")
if files:
    run = json.loads(files[0].read_text())
    print(f"\n--- First run ---")
    meta = run['metadata']
    print(f"product: {meta['product']}")
    print(f"v={meta['v']:.2f}  quantity={meta['quantity']}  initial={meta['seller_config']['initial_strategy']}  adaptive={meta['seller_config']['adaptive']}")
    for r in run['rounds']:
        sp = r.get('supplier_price')
        sp_str = f"${sp:.2f}" if sp else "FAIL"
        print(f"  R{r['round']:>2}  m={r['buyer_price']:.2f}  s={sp_str}  strat={r['supplier_active_strategy']}")
